In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 14 Lab: Verification Techniques, AI-Driven Testing & PPA Analysis

## Overview
The capstone verification session — all four cross-cutting threads converge.
You'll add assertions to existing designs, implement a constraint-based UART
parity extension, generate AI-driven testbenches for your project, and produce
a structured PPA analysis.

## Prerequisites
- Pre-class videos on assertions, AI verification workflows, PPA methodology, and coverage
- Working UART TX (SV version from Day 13 or Verilog from Day 11)
- Working FSM from Day 7 or Day 13
- Final project module (at least the interface defined) for Exercise 3

## Toolchain Notes
- Immediate assertions: supported by `iverilog -g2012`
- Concurrent assertions: NOT supported by Icarus (write as documentation only)
- Covergroups: NOT supported by Icarus (manual analysis in Exercise 4)

## Exercises

| # | Exercise | Time | Key SLOs |
|---|----------|------|----------|
| 1 | Assertion-Enhanced UART TX | 25 min | 14.1 |
| 2 | Constraint-Based UART Parity Extension | 20 min | 14.2, 14.4 |
| 3 | AI Constraint-Based TB for Project Module | 25 min | 14.3, 14.6 |
| 4 | PPA Analysis Exercise | 25 min | 14.4, 14.5 |
| 5 | Project Work Time | 15 min | — |

> **Instructor note:** Exercise 2 has an escape valve — students behind on their
> final project may skip it and use the time for Exercises 3–4. Parity can be
> completed as homework.

## Deliverables

1. **Assertion-enhanced UART TX** with 7 assertions + bug injection demo
2. **Parity-parameterized UART TX** with PPA comparison (PARITY_EN=0 vs 1)
3. **AI constraint-based TB** with: constraint spec + raw AI output + corrected TB + coverage analysis
4. **PPA analysis report** — comparison table with real data + 2 analysis paragraphs

## Supplementary Material

The following exercises from the original Day 14 build are available as supplementary
content for students who finish early or want additional practice:

- `supplementary/fsm_assertions/` — Add transition and output assertions to the traffic light FSM
- `supplementary/coverage_worksheet/` — Manual functional coverage analysis for the ALU

## Shared Resources
- `go_board.pcf` — Pin constraint file
- Reuse modules from `shared/lib/` for PPA analysis targets

---
## Exercise Files

The starter files for each exercise are below. Edit the code, then run the simulation/build cells.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile uart_tx_asserted.sv
// =============================================================================
// uart_tx_asserted.sv — UART TX with Assertion Layer
// Day 14, Exercise 1
// =============================================================================
// Start from the SV UART TX and add 7 assertions.

module uart_tx_asserted #(
    parameter int CLK_FREQ  = 25_000_000,
    parameter int BAUD_RATE = 115_200
)(
    input  logic       i_clk, i_reset, i_valid,
    input  logic [7:0] i_data,
    output logic       o_tx, o_busy
);
    localparam int CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam int BAUD_CNT_W   = $clog2(CLKS_PER_BIT);

    typedef enum logic [1:0] {
        S_IDLE=2'd0, S_START=2'd1, S_DATA=2'd2, S_STOP=2'd3
    } uart_state_t;

    uart_state_t state;
    logic [BAUD_CNT_W-1:0] baud_cnt;
    logic [7:0] shift_reg;
    logic [2:0] bit_idx;

    logic baud_tick;
    assign baud_tick = (baud_cnt == BAUD_CNT_W'(CLKS_PER_BIT - 1));
    assign o_busy = (state != S_IDLE);

    always_ff @(posedge i_clk) begin
        if (i_reset) begin
            state <= S_IDLE; o_tx <= 1'b1;
            baud_cnt <= '0; bit_idx <= '0; shift_reg <= '0;
        end else begin
            case (state)
                S_IDLE: begin
                    o_tx <= 1'b1; baud_cnt <= '0; bit_idx <= '0;
                    if (i_valid) begin
                        shift_reg <= i_data; state <= S_START;
                    end
                end
                S_START: begin
                    o_tx <= 1'b0; baud_cnt <= baud_cnt + 1;
                    if (baud_tick) begin baud_cnt <= '0; state <= S_DATA; end
                end
                S_DATA: begin
                    o_tx <= shift_reg[0]; baud_cnt <= baud_cnt + 1;
                    if (baud_tick) begin
                        baud_cnt <= '0;
                        shift_reg <= {1'b0, shift_reg[7:1]};
                        if (bit_idx == 3'd7) state <= S_STOP;
                        else bit_idx <= bit_idx + 1;
                    end
                end
                S_STOP: begin
                    o_tx <= 1'b1; baud_cnt <= baud_cnt + 1;
                    if (baud_tick) begin baud_cnt <= '0; state <= S_IDLE; end
                end
            endcase
        end
    end

    // =================================================================
    // ASSERTIONS — Add all 7 below
    // =================================================================

    // TODO Assertion 1: TX line must be HIGH when in IDLE state
    // always_comb begin
    //     if (state == S_IDLE)
    //         assert (o_tx == 1'b1)
    //             else $error("A1: TX not high in IDLE");
    // end

    // ---- YOUR ASSERTION 1 HERE ----

    // TODO Assertion 2: o_busy must equal (state != S_IDLE)
    // ---- YOUR ASSERTION 2 HERE ----

    // TODO Assertion 3: bit_idx must be in range [0, 7]
    // ---- YOUR ASSERTION 3 HERE ----

    // TODO Assertion 4: baud_cnt must be in range [0, CLKS_PER_BIT-1]
    // ---- YOUR ASSERTION 4 HERE ----

    // TODO Assertion 5: When state == S_START, o_tx must be 0
    // ---- YOUR ASSERTION 5 HERE ----

    // TODO Assertion 6: When state == S_STOP, o_tx must be 1
    // ---- YOUR ASSERTION 6 HERE ----

    // TODO Assertion 7: If o_busy, i_valid should not be asserted
    //   (Use $warning — this is the caller's responsibility)
    // ---- YOUR ASSERTION 7 HERE ----

endmodule

In [ ]:
%%writefile Makefile
# Makefile — uart_tx_asserted
PROJECT  = uart_tx_asserted
TOP      = uart_tx_asserted
PCF      = ../go_board.pcf
SRCS     = uart_tx_asserted.sv
TB       = tb_uart_tx_asserted.sv

DEVICE   = hx1k
PACKAGE  = vq100

IVERILOG_FLAGS = -g2012 -Wall

all: sim

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(filter-out $(TB),$(SRCS))
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: all prog sim wave stat clean

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile tb_uart_tx_parity.sv
// =============================================================================
// tb_uart_tx_parity.sv — Testbench for UART TX with Parity
// Day 14, Exercise 2
// =============================================================================
// Tests both PARITY_EN=0 and PARITY_EN=1 configurations.
// Verifies frame structure: start + 8 data + [parity] + stop
`timescale 1ns/1ps

module tb_uart_tx_parity;

    localparam int CLK_FREQ    = 25_000_000;
    localparam int BAUD_RATE   = 115_200;
    localparam int CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam int CLK_PERIOD   = 40;  // 25 MHz = 40 ns

    logic       clk, reset, valid;
    logic [7:0] data;
    logic       tx_no_par, busy_no_par;
    logic       tx_even,   busy_even;

    // Instance 1: No parity
    uart_tx_parity #(
        .CLK_FREQ(CLK_FREQ), .BAUD_RATE(BAUD_RATE),
        .PARITY_EN(0)
    ) uut_no_parity (
        .i_clk(clk), .i_reset(reset), .i_valid(valid), .i_data(data),
        .o_tx(tx_no_par), .o_busy(busy_no_par)
    );

    // Instance 2: Even parity
    uart_tx_parity #(
        .CLK_FREQ(CLK_FREQ), .BAUD_RATE(BAUD_RATE),
        .PARITY_EN(1), .PARITY_TYPE(0)
    ) uut_even_parity (
        .i_clk(clk), .i_reset(reset), .i_valid(valid), .i_data(data),
        .o_tx(tx_even), .o_busy(busy_even)
    );

    always #(CLK_PERIOD/2) clk = ~clk;

    integer pass_count = 0, fail_count = 0;

    task send_byte(input [7:0] d);
        begin
            data = d;
            @(posedge clk);
            valid = 1;
            @(posedge clk);
            valid = 0;
            // Wait for both instances to finish
            wait (!busy_no_par && !busy_even);
            repeat(CLKS_PER_BIT) @(posedge clk);  // gap between frames
        end
    endtask

    // Verify no-parity TX line idle is high
    task check_idle;
        begin
            if (tx_no_par !== 1'b1) begin
                $display("FAIL: no-parity TX not idle high"); fail_count++;
            end else pass_count++;
            if (tx_even !== 1'b1) begin
                $display("FAIL: even-parity TX not idle high"); fail_count++;
            end else pass_count++;
        end
    endtask

    initial begin
        $dumpfile("uart_parity.vcd");
        $dumpvars(0, tb_uart_tx_parity);

        clk = 0; reset = 1; valid = 0; data = 0;
        repeat(5) @(posedge clk);
        reset = 0;
        repeat(3) @(posedge clk);

        check_idle();

        // Send test bytes
        $display("Sending 0x41 ('A')...");
        send_byte(8'h41);

        $display("Sending 0x00...");
        send_byte(8'h00);

        $display("Sending 0xFF...");
        send_byte(8'hFF);

        $display("Sending 0xAA...");
        send_byte(8'hAA);

        $display("Sending 0x55...");
        send_byte(8'h55);

        repeat(100) @(posedge clk);

        $display("");
        $display("========================================");
        $display("  Basic frame timing tests: %0d passed, %0d failed", pass_count, fail_count);
        $display("  Visual verification: check VCD for frame structure");
        $display("  No-parity frame: 10 bit periods (start+8data+stop)");
        $display("  Even-parity frame: 11 bit periods (start+8data+parity+stop)");
        $display("========================================");

        $finish;
    end

endmodule

In [ ]:
%%writefile uart_tx_parity.sv
// =============================================================================
// uart_tx_parity.sv — UART TX with Configurable Parity
// Day 14, Exercise 2: Constraint-Based Design with generate if
// =============================================================================
// Add configurable parity to the UART TX module using generate if.
//
// When PARITY_EN=0: standard 10-bit frame (start + 8 data + stop)
// When PARITY_EN=1: 11-bit frame (start + 8 data + parity + stop)
//
// PARITY_TYPE=0: even parity (XOR reduction of data)
// PARITY_TYPE=1: odd parity  (inverted XOR reduction)
//
// After implementing, synthesize both configurations and compare with
// yosys stat. Record the PPA delta in the ppa_worksheet.md.
// =============================================================================

module uart_tx_parity #(
    parameter int CLK_FREQ    = 25_000_000,
    parameter int BAUD_RATE   = 115_200,
    parameter int PARITY_EN   = 0,   // 0 = no parity, 1 = parity enabled
    parameter int PARITY_TYPE = 0    // 0 = even, 1 = odd (only used when PARITY_EN=1)
)(
    input  logic       i_clk,
    input  logic       i_reset,
    input  logic       i_valid,
    input  logic [7:0] i_data,
    output logic       o_tx,
    output logic       o_busy
);

    localparam int CLKS_PER_BIT = CLK_FREQ / BAUD_RATE;
    localparam int BAUD_CNT_W   = $clog2(CLKS_PER_BIT);

    // FSM states
    // TODO: Add a S_PARITY state between S_DATA and S_STOP when parity is enabled.
    // Hint: You can define the enum with S_PARITY always present, but only
    //       transition through it when PARITY_EN==1.
    typedef enum logic [2:0] {
        S_IDLE  = 3'd0,
        S_START = 3'd1,
        S_DATA  = 3'd2,
        S_PARITY= 3'd3,   // used only when PARITY_EN=1
        S_STOP  = 3'd4
    } state_t;

    state_t state;
    logic [BAUD_CNT_W-1:0] baud_cnt;
    logic [7:0] shift_reg;
    logic [2:0] bit_idx;

    logic baud_tick;
    assign baud_tick = (baud_cnt == BAUD_CNT_W'(CLKS_PER_BIT - 1));
    assign o_busy = (state != S_IDLE);

    // ================================================================
    // Parity calculation — use generate if to conditionally include
    // ================================================================
    // TODO: Use generate if (PARITY_EN) to create the parity logic.
    //   Even parity: XOR reduction of the data byte (^i_data)
    //   Odd parity:  inverted XOR reduction (~^i_data)
    //
    // generate
    //     if (PARITY_EN) begin : gen_parity
    //         logic parity_bit;
    //         if (PARITY_TYPE == 0)
    //             assign parity_bit = ^i_data;       // even
    //         else
    //             assign parity_bit = ~(^i_data);    // odd
    //     end
    // endgenerate

    // ---- YOUR PARITY GENERATION HERE ----


    // ================================================================
    // Main FSM
    // ================================================================
    always_ff @(posedge i_clk) begin
        if (i_reset) begin
            state     <= S_IDLE;
            o_tx      <= 1'b1;
            baud_cnt  <= '0;
            bit_idx   <= '0;
            shift_reg <= '0;
        end else begin
            case (state)
                S_IDLE: begin
                    o_tx <= 1'b1;
                    baud_cnt <= '0;
                    bit_idx  <= '0;
                    if (i_valid) begin
                        shift_reg <= i_data;
                        state     <= S_START;
                    end
                end

                S_START: begin
                    o_tx <= 1'b0;  // start bit
                    baud_cnt <= baud_cnt + 1;
                    if (baud_tick) begin
                        baud_cnt <= '0;
                        state    <= S_DATA;
                    end
                end

                S_DATA: begin
                    o_tx <= shift_reg[0];
                    baud_cnt <= baud_cnt + 1;
                    if (baud_tick) begin
                        baud_cnt  <= '0;
                        shift_reg <= {1'b0, shift_reg[7:1]};
                        if (bit_idx == 3'd7) begin
                            // TODO: Transition to S_PARITY when PARITY_EN=1,
                            //       or S_STOP when PARITY_EN=0.
                            // Hint: Use generate if to select the next state,
                            //       or use a simple if on the parameter
                            //       (parameters are compile-time constants).

                            state <= S_STOP;  // ← fix this for parity support

                        end else begin
                            bit_idx <= bit_idx + 1;
                        end
                    end
                end

                // TODO: Implement S_PARITY state
                // When PARITY_EN=1: drive o_tx with the parity bit for one bit period,
                // then transition to S_STOP.
                // When PARITY_EN=0: this state is never entered (dead code, optimizer removes it).

                // ---- YOUR S_PARITY STATE HERE ----


                S_STOP: begin
                    o_tx <= 1'b1;  // stop bit
                    baud_cnt <= baud_cnt + 1;
                    if (baud_tick) begin
                        baud_cnt <= '0;
                        state    <= S_IDLE;
                    end
                end

                default: state <= S_IDLE;
            endcase
        end
    end

endmodule

In [ ]:
%%writefile Makefile
# Makefile — UART TX Parity Extension
# Day 14, Exercise 2

SRCS     = uart_tx_parity.sv
TB       = tb_uart_tx_parity.sv
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVERILOG_FLAGS = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave uart_parity.vcd &

# PPA comparison: synthesize both configurations
stat_no_parity:
	@echo "=== PARITY_EN=0 ==="
	yosys -p "read_verilog -sv -DPARITY_EN=0 $(SRCS); synth_ice40 -top uart_tx_parity; stat"

stat_even_parity:
	@echo "=== PARITY_EN=1, PARITY_TYPE=0 (even) ==="
	yosys -p "read_verilog -sv $(SRCS); synth_ice40 -top uart_tx_parity; stat"

stat_both: stat_no_parity stat_even_parity

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave stat_no_parity stat_even_parity stat_both clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_uart_tx_parity.sv uart_tx_parity.sv && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile Makefile
# Makefile — AI Constraint-Based Testbench
# Day 14, Exercise 3
#
# Students: set TOP, SRCS, and TB to your project module files.

TOP    ?= your_module
SRCS   ?= your_module.sv
TB     ?= tb_your_module.sv
IVERILOG_FLAGS = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVERILOG_FLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp
	@echo "=== Simulation complete ==="

wave: sim
	gtkwave *.vcd &

clean:
	rm -f *.vvp *.vcd *.log

.PHONY: sim wave clean

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile Makefile
# Makefile — PPA Analysis Helpers
# Day 14, Exercise 4
#
# Quick synthesis stat targets for common modules.
# Adjust paths to match your repo layout.

SHARED = ../../../../shared/lib

# Counter at multiple widths
stat_counter_8:
	yosys -q -p "read_verilog -DWIDTH=8 $(SHARED)/counter_mod_n.v; synth_ice40 -top counter_mod_n; stat"

stat_counter_16:
	yosys -q -p "read_verilog -DWIDTH=16 $(SHARED)/counter_mod_n.v; synth_ice40 -top counter_mod_n; stat"

stat_counter_32:
	yosys -q -p "read_verilog -DWIDTH=32 $(SHARED)/counter_mod_n.v; synth_ice40 -top counter_mod_n; stat"

# UART TX from shared lib
stat_uart_tx:
	yosys -q -p "read_verilog $(SHARED)/uart_tx.v; synth_ice40 -top uart_tx; stat"

# UART TX with parity (from exercise 2)
stat_uart_parity_off:
	yosys -q -p "read_verilog -sv -DPARITY_EN=0 ../ex2_uart_parity/starter/uart_tx_parity.sv; synth_ice40 -top uart_tx_parity; stat"

stat_uart_parity_on:
	yosys -q -p "read_verilog -sv ../ex2_uart_parity/starter/uart_tx_parity.sv; synth_ice40 -top uart_tx_parity; stat"

# Run all
stat_all: stat_counter_8 stat_counter_16 stat_counter_32 stat_uart_tx stat_uart_parity_off stat_uart_parity_on

clean:
	rm -f *.json *.log

.PHONY: stat_counter_8 stat_counter_16 stat_counter_32 stat_uart_tx stat_uart_parity_off stat_uart_parity_on stat_all clean